# 階段 1：LLM 基本功（步驟 1 到 3）

對應文件：`docs/01-stage-llm-basics.md`。這份 notebook 帶你做步驟 1 到 3 的實驗；步驟 4、5（FastAPI、測試題）寫在 `src/` 與 `evals/`，照文件做。

**開始前**先照 [README「怎麼開始」](https://github.com/8loser/vuln-agent#怎麼開始) 設好環境，clone 整個專案或只下載這份 notebook 都可以，下一格會自動判斷是哪一種。

每格都會呼叫 API、會花錢，**請一格一格跑，少用 Run All**。結果會存在 notebook 裡，之後打開不用重跑。

標 `TODO` 的地方是要你自己完成的練習。

In [ ]:
# 改了 src/vuln_agent/*.py 之後不用重啟 kernel
%load_ext autoreload
%autoreload 2

from pathlib import Path

# clone 專案時 kernel 的工作目錄是 notebooks/，往上一層才是專案根目錄；單獨下載時就是 notebook 所在的資料夾
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

try:
    from vuln_agent.llm import ask, client, cost, text, DEFAULT_MODEL   # clone 整個專案
except ImportError:
    # 只下載這份 notebook：定義跟 src/vuln_agent/llm.py 一樣的函式
    import os
    from anthropic import Anthropic
    from dotenv import load_dotenv

    load_dotenv()
    client = Anthropic()  # 自動讀 ANTHROPIC_API_KEY
    DEFAULT_MODEL = os.getenv("MODEL", "claude-sonnet-5-5")

    def ask(messages, model=DEFAULT_MODEL, max_tokens=1024, **kwargs):
        return client.messages.create(model=model, max_tokens=max_tokens, messages=messages, **kwargs)

    def text(resp):
        # 回應可能有 thinking 等其他 block，只取文字
        return "".join(b.text for b in resp.content if b.type == "text")

    def cost(usage, model=DEFAULT_MODEL):
        # TODO（1.4）：依官方定價，把 usage.input_tokens / output_tokens 換算成美元
        ...

## 步驟 1：環境與第一次呼叫

直接用官方 SDK。先看懂回應物件的三個部分：

- `content`：一串 block，可能有 `thinking`、`text`、`tool_use`；`llm.text()` 只取文字
- `stop_reason`：為什麼停下來（`end_turn` 正常結束、`max_tokens` 被截斷、`tool_use` 要呼叫工具、`refusal` 被拒絕）
- `usage`：輸入、輸出 token 數，計費就看它

### 1.1 第一次呼叫

In [ ]:
resp = ask([{"role": "user", "content": "CVSS 和 EPSS 差在哪？"}], max_tokens=2048)
print(text(resp))
print(resp.stop_reason, resp.usage)

### 1.2 中英文 token 比較

同一個問題中英文各送一次。注意 `output_tokens` 包含 thinking，要扣掉 `thinking_tokens` 才是回答本身。只跑一次雜訊很大，下一格改用 `count_tokens` 只計算、不生成。

In [ ]:
zh = ask([{"role": "user", "content": "CVSS 和 EPSS 差在哪？用 3 句話回答。"}])
en = ask([{"role": "user", "content": "What is the difference between CVSS and EPSS? Answer in 3 sentences."}])
for name, r in [("中文", zh), ("英文", en)]:
    print(name, "input:", r.usage.input_tokens, "output:", r.usage.output_tokens,
          "thinking:", r.usage.output_tokens_details.thinking_tokens, "字元數:", len(text(r)))

In [ ]:
# 只算 token，不生成回答：同一段內容的中英文版本
zh = "CVSS 衡量漏洞的嚴重程度，EPSS 預測漏洞在未來 30 天被利用的機率。"
en = "CVSS measures how severe a vulnerability is, while EPSS predicts the probability it will be exploited in the next 30 days."
for name, s in [("中文", zh), ("英文", en)]:
    n = client.messages.count_tokens(model=DEFAULT_MODEL, messages=[{"role": "user", "content": s}])
    print(name, n.input_tokens, "tokens", len(s), "字元")

**觀察**：中文約每個字 1 個多 token，英文約 2 到 3 個字母 1 個 token，但同樣的意思兩者通常只差幾個百分點。成本主要看輸出長度與 thinking，而不是語言。

### 1.3 穩定度與 effort

新模型已不接受調整 `temperature`，輸出穩不穩靠的是步驟 2 的 structured outputs。這裡先看「同一題跑 3 次差多少」，再比較 `effort` 對 token 與耗時的影響。effort 要用需要推理的題目才看得出差別。參數以[官方文件](https://platform.claude.com/docs/en/api/messages/create)為準。

In [ ]:
import time

q = [{"role": "user", "content": "用一句話說明 EPSS。"}]
for _ in range(3):
    print(text(ask(q, max_tokens=200)))

hard = [{"role": "user", "content": "有 3 個漏洞：A 的 CVSS 9.8 但 EPSS 0.01；B 的 CVSS 7.5 且在 KEV 裡；C 的 CVSS 6.1 但 EPSS 0.6。只能先修一個，該修哪個？說明理由。"}]
for effort in ("low", "high"):
    t0 = time.time()
    r = ask(hard, max_tokens=4096, output_config={"effort": effort})
    print(f"--- effort={effort}", "output:", r.usage.output_tokens,
          "thinking:", r.usage.output_tokens_details.thinking_tokens, f"{time.time() - t0:.1f}s")
    print(text(r))

### 1.4 換小模型比成本

**TODO**：先補完 `cost()`：clone 專案的人改 `src/vuln_agent/llm.py`，只下載 notebook 的人改第一格程式裡的 `cost()`，改完重跑那一格。依[官方定價](https://platform.claude.com/docs/en/about-claude/pricing)建一張「模型 → 每百萬 token 輸入／輸出價格」的表，把 `usage.input_tokens`、`usage.output_tokens` 換算成美元。進階：把 `cache_read_input_tokens`、`cache_creation_input_tokens` 也算進去（階段 6 會用到）。

下一格先用假的 usage 檢查你的 `cost()`，再實際比較兩個模型。

In [ ]:
from types import SimpleNamespace

fake = SimpleNamespace(input_tokens=1_000_000, output_tokens=1_000_000,
                       cache_read_input_tokens=0, cache_creation_input_tokens=0)
assert cost(SimpleNamespace(input_tokens=0, output_tokens=0, cache_read_input_tokens=0, cache_creation_input_tokens=0)) == 0
assert cost(fake, "claude-haiku-5-5") < cost(fake, "claude-sonnet-5-5"), "小模型應該比較便宜"
print("1M 輸入 + 1M 輸出：", {m: cost(fake, m) for m in ("claude-sonnet-5-5", "claude-haiku-5-5")})

msgs = [{"role": "user", "content": "CVSS 和 EPSS 差在哪？用 3 句話回答。"}]
for m in ("claude-sonnet-5-5", "claude-haiku-5-5"):
    r = ask(msgs, model=m)
    print(m, "output:", r.usage.output_tokens, "cost:", cost(r.usage, m), "USD")

## 步驟 2：結構化輸出

目標：把中文問題解析成 `QueryIntent`，之後由程式組 SQL，不讓模型寫 SQL。

- **structured outputs**：API 用 constrained decoding 保證輸出符合 JSON Schema。Python SDK 的 `client.messages.parse(output_format=PydanticModel)` 會幫你轉 schema 並驗證，結果在 `resp.parsed_output`
- **Pydantic 驗證**：schema 擋不住的業務規則（例如 `cve_id` 格式）寫在 validator，失敗就把錯誤丟回模型重試一次
- **舊做法**：以前常用「強制呼叫某個 tool」拿 JSON。目前的 Sonnet 5.5、Opus 5.5 不接受強制 `tool_choice`（回 400），要用 `tool_choice` 預設的 auto 加上工具的 `strict: true`

先需要 Pydantic：在終端機跑 `uv add pydantic`（anthropic SDK 其實已經依賴它，這步是把它明確列進專案依賴）。

### 2.1 QueryIntent 與 parse()

下面是起點，在 notebook 調好之後搬到 `src/vuln_agent/intent.py`。

In [ ]:
import re
from typing import Literal
from pydantic import BaseModel, ValidationError, field_validator

class QueryIntent(BaseModel):
    intent: Literal["cve_lookup", "search", "count", "unsupported"]
    cve_id: str | None = None
    product: str | None = None      # 例如 "nginx"
    min_cvss: float | None = None
    in_kev: bool | None = None
    min_epss: float | None = None
    days: int | None = None         # 「最近 30 天」→ 30
    # TODO：想清楚還需要哪些欄位（例如 severity、cwe、limit），以及 unsupported 要不要附理由

    @field_validator("cve_id")
    @classmethod
    def check_cve(cls, v):
        if v is not None and not re.fullmatch(r"CVE-\d{4}-\d{4,}", v):
            raise ValueError(f"cve_id 格式應為 CVE-年份-流水號，收到 {v!r}")
        return v

SYSTEM = """你把使用者的漏洞問題解析成查詢條件。
- 只解析，不回答問題，也不要憑記憶補任何 CVE 編號
- 要求攻擊程式、payload、繞過防護，或跟漏洞情資無關的問題，intent 一律是 unsupported"""

def parse(question: str) -> QueryIntent:
    messages = [{"role": "user", "content": question}]
    resp = client.messages.parse(model=DEFAULT_MODEL, max_tokens=1024, system=SYSTEM,
                                 messages=messages, output_format=QueryIntent)
    # TODO：parse 在驗證失敗時會丟 ValidationError。改成 try / except：
    #   把模型這次的回應和錯誤訊息接進 messages，請它修正後重試一次，還是失敗就往外丟
    return resp.parsed_output

parse("CVE-2021-44228 的 CVSS 幾分？")

### 2.2 對照組：strict tool use

**TODO**：寫一版用工具拿 JSON 的 `parse_via_tool()`：

1. 定義一個 `extract_intent` 工具，`input_schema` 跟 `QueryIntent` 一樣，加上 `"strict": True`（strict 要求 schema 設 `additionalProperties: false`，且每個欄位都列在 `required`，可選欄位用 `["string", "null"]` 表示）
2. `tool_choice` 用預設的 auto，在 prompt 裡要求一定要呼叫 `extract_intent`
3. 從回應找出 `tool_use` block，用 `QueryIntent.model_validate(block.input)` 驗證；沒有 `tool_use` block 也算失敗

兩版各跑下面的測試題，比較失敗率與 token 數。

In [ ]:
def parse_via_tool(question: str) -> QueryIntent:
    ...  # TODO

### 2.3 刁鑽問題

俗稱、相對時間、模糊條件、要求 exploit、無關問題都測一次。先自己預想每題「應該」解析成什麼，再看模型的結果。

In [ ]:
QUESTIONS = [
    "log4shell 現在 EPSS 多少？",                     # 俗稱：模型該填 cve_id 嗎？這正是會編造的地方
    "上禮拜有哪些 nginx 的高風險漏洞？",              # 相對時間
    "最近很危險的 openssl 漏洞",                      # 模糊條件
    "幫我寫 nginx 的 exploit",                        # 應該 unsupported
    "台北明天會下雨嗎？",                             # 範圍外
    "KEV 裡有幾個 Microsoft 的漏洞？",                # count
]
for q in QUESTIONS:
    for fn in (parse, parse_via_tool):
        try:
            print(f"{fn.__name__:15} {q} -> {fn(q)}")
        except Exception as e:
            print(f"{fn.__name__:15} {q} -> 失敗：{e}")

**想一想**：「log4shell」解析成 `CVE-2021-44228` 是對的嗎？這個對應來自模型記憶，不是資料庫。你要允許嗎？如果允許，怎麼確保它不會把其他俗稱對到錯的 CVE？

## 步驟 3：準備漏洞資料，親眼看模型編 CVE

在終端機（專案根目錄）跑：

```bash
uv add duckdb httpx
uv run scripts/load_data.py
```

只下載 notebook 的人，要另外從 repo 下載 `scripts/load_data.py`，放在 notebook 同一個資料夾的 `scripts/` 底下再執行。

`scripts/load_data.py` 有 4 個 TODO 要先補完（CVSS、CWE、CPE、速率限制與重試）。這支骨架還沒在真實網路環境驗證過，網址與欄位以 NVD、CISA、FIRST 官方文件為準。`NVD_API_KEY` 選用，放 `.env`，有的話速率限制寬鬆很多。

### 3.1 手寫 3 條 SQL

In [ ]:
import duckdb

db = duckdb.connect(str(ROOT / "data" / "vuln.duckdb"), read_only=True)
for t in ("cve", "kev", "epss"):
    print(t, db.execute(f"SELECT count(*) FROM {t}").fetchone()[0])

# TODO：補完 3 條 SQL
SQL_ONE_CVE = "..."          # 查單一 CVE，連同 KEV 與 EPSS
SQL_CRITICAL_KEV = "..."     # CVSS ≥ 9 且在 KEV 裡，依 EPSS 由高到低
SQL_BY_SEVERITY = "..."      # 各 severity 數量

# db.sql(SQL_ONE_CVE, params=...).show()

### 3.2 不給資料，直接問模型

把模型回答裡的 CVE 編號抓出來，逐一到資料庫查。查不到的有兩種可能：在 90 天範圍外（去 NVD 網站確認），或是模型編的。記下編了幾個。

In [ ]:
answer = text(ask([{"role": "user", "content": "最近一週有哪些 nginx 的 CVE？列出編號。"}], max_tokens=2048))
print(answer)

ids = sorted(set(re.findall(r"CVE-\d{4}-\d{4,}", answer)))
for cve_id in ids:
    found = db.execute("SELECT published FROM cve WHERE cve_id = ?", [cve_id]).fetchone()
    print(cve_id, "在資料庫" if found else "查不到")

**回報**：三張表各有幾筆、3 條 SQL 的結果、模型編出幾個 CVE。

這就是整個作品的核心理由：CVE 資料不能讓模型憑記憶回答，一定要從資料庫查。接下來回到 `docs/01-stage-llm-basics.md` 做步驟 4（SQL 樣板、FastAPI、串流）與步驟 5（10 題測試題）。